In [1]:
@file:DependsOn("io.ktor:ktor-client-core:2.3.7")
@file:DependsOn("io.ktor:ktor-client-cio:2.3.7")
@file:DependsOn("io.ktor:ktor-client-content-negotiation:2.3.7")
@file:DependsOn("io.ktor:ktor-serialization-kotlinx-json:2.3.7")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-serialization-json:1.6.2")

import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.client.plugins.contentnegotiation.*
import io.ktor.http.*
import kotlinx.serialization.*
import kotlinx.serialization.json.*

### Models

In [3]:
@Serializable
data class RunConfig(
    val baseUrl: String,          // e.g. "https://api.openai.com/v1"
    val apiKey: String,           // set from env
    val model: String,            // e.g. "gpt-4.1-mini" or whatever
    val temperature: Double = 0.2,
    val maxTokens: Int = 600
)

@Serializable
data class Message(val role: String, val content: String)

@Serializable
data class ChatRequest(
    val model: String,
    val messages: List<Message>,
    val temperature: Double,
    @SerialName("max_tokens") val maxTokens: Int
)

@Serializable
data class ChatResponse(
    val choices: List<Choice>
) {
    @Serializable data class Choice(val message: ChoiceMessage)
    @Serializable data class ChoiceMessage(val role: String, val content: String)
}

data class HopResult(
    val seed: String,
    val expandedEmail: String,
    val summary: String
)

### HTTP client + “chat completions” call

This assumes an OpenAI-style endpoint at POST /chat/completions.

In [4]:
import io.ktor.serialization.kotlinx.json.*

val jsonConfig = Json {
    ignoreUnknownKeys = true
    prettyPrint = false
}

val http = HttpClient(CIO) {
    install(ContentNegotiation) { json(jsonConfig) }
}

suspend fun chatCompletion(cfg: RunConfig, messages: List<Message>): String {
    val url = "${cfg.baseUrl.trimEnd('/')}/chat/completions"
    val req = ChatRequest(
        model = cfg.model,
        messages = messages,
        temperature = cfg.temperature,
        maxTokens = cfg.maxTokens
    )

    val resp: HttpResponse = http.post(url) {
        contentType(ContentType.Application.Json)
        header(HttpHeaders.Authorization, "Bearer ${cfg.apiKey}")
        setBody(req)
    }

    val bodyText = resp.bodyAsText()
    if (resp.status.value !in 200..299) {
        throw RuntimeException("HTTP ${resp.status.value}: $bodyText")
    }

    val parsed = jsonConfig.decodeFromString<ChatResponse>(bodyText)
    return parsed.choices.firstOrNull()?.message?.content
        ?: throw RuntimeException("No choices in response: $bodyText")
}

### Prompt templates (locked protocol)

In [ ]:
fun expandPrompt(seed: String, tone: String = "professional, clear, not overly long"): List<Message> =
    listOf(
        Message("system", "You are an assistant that rewrites messages into complete emails."),
        Message("user",
            """
      Rewrite the following message as a complete email.
      Rules:
      - Preserve ALL factual content (names, dates, numbers, requests).
      - Do NOT add new facts or backstory.
      - Tone: $tone
      - Output ONLY the email body (no commentary).

      Message:
      $seed
      """.trimIndent()
        )
    )

fun summarizePrompt(email: String): List<Message> =
    listOf(
        Message(
            "system",
            """You summarize emails.
Rules:
- Always summarize the content of the email.
- Do not add new facts.
- Do not comment on what is missing (e.g., do not say 'no requests were mentioned').
- If the email contains requests/dates/numbers/constraints, include them; otherwise just summarize the main message.
Output only the summary.""".trimIndent()
        ),
        Message(
            "user",
            """
Summarize the following email into 1–3 sentences.

If present, preserve:
- requests/actions
- deadlines/dates/times
- names/numbers
- constraints (e.g., "do not mention X")

Rules:
- Do NOT add or infer anything.
- Do NOT list fields that are absent.
- Output ONLY the summary.

Email:
$email
""".trimIndent()
        )
    )

In [ ]:
// === Prompt templates ===

fun expandPromptChaos(seed: String): List<Message> =
    listOf(
        Message("system", "You write emails. Make them natural, human, and complete."),
        Message(
            "user",
            """
Take this short message and expand it into a full email.
- Feel free to add reasonable context, assumptions, and details.
- Make it sound realistic and socially smooth.
- Include a subject line.
- Output only the email.

Message:
$seed
""".trimIndent()
        )
    )

fun summarizePromptChaos(email: String): List<Message> =
    listOf(
        Message("system", "You summarize emails."),
        Message(
            "user",
            """
Summarize the email in your own words.
- Be concise but keep what you think matters.
- You may paraphrase, interpret intent, and drop minor details.
- Output only the summary.

Email:
$email
""".trimIndent()
        )
    )

fun expandPrompt(seed: String, tone: String = "professional, clear, not overly long"): List<Message> =
    listOf(
        Message("system", "You are an assistant that rewrites messages into complete emails."),
        Message(
            "user",
            """
Rewrite the following message as a complete email.

Rules:
- Preserve ALL factual content (names, dates, numbers, requests).
- Do NOT add new facts or backstory.
- Tone: $tone
- Output ONLY the email body (no commentary).

Message:
$seed
""".trimIndent()
        )
    )

fun summarizePrompt(email: String): List<Message> =
    listOf(
        Message(
            "system",
            """
You summarize emails.

Rules:
- Always summarize the content of the email.
- Do not add new facts.
- Do not comment on what is missing (e.g., do not say "no requests were mentioned").
- If the email contains requests/dates/numbers/constraints, include them; otherwise just summarize the main message.
- Output ONLY the summary.
""".trimIndent()
        ),
        Message(
            "user",
            """
Summarize the following email into 1–3 sentences.

If present, preserve:
- requests/actions
- deadlines/dates/times
- names/numbers
- constraints (e.g., "do not mention X")

Rules:
- Do NOT add or infer anything.
- Do NOT list fields that are absent.
- Output ONLY the summary.

Email:
$email
""".trimIndent()
        )
    )

In [5]:
fun expandPrompt(seed: String): List<Message> =
    listOf(
        Message("system", "You write emails."),
        Message("user",
            """
      Take this short message and expand it into a full email.
      - Feel free to add assumptions and expand on them
      - Be creative with and elaborate on details
      - Make it sound hip, cool, and interesting
      - Include a subject line.
      - Output only the email.

      Message:
      $seed
      """.trimIndent()
        )
    )

fun summarizePrompt(email: String): List<Message> =
    listOf(
        Message("system", "You summarize emails."),
        Message("user",
            """
      Summarize the email in your own words in a few sentences.

      - Output only the summary.

      Email:
      $email
      """.trimIndent()
        )
    )

### Run one hop

In [8]:
var seed = "It rained all day and we didn't get any work done."


In [ ]:
var seed = "Beth burnt her hair on the toaster and now she's suing the company."

In [9]:

val cfg = RunConfig(
    baseUrl = System.getenv("OPENAI_BASE_URL") ?: "https://api.openai.com/v1",
    apiKey  = System.getenv("OPENAI_API_KEY") ?: error("Set OPENAI_API_KEY in your env"),
    model   = System.getenv("OPENAI_MODEL") ?: "gpt-4.1-mini",
    temperature = 0.2,
    maxTokens = 600
)

suspend fun runOneHop(seed: String, cfg: RunConfig): HopResult {
    val expanded = chatCompletion(cfg, expandPrompt(seed))
    val summary  = chatCompletion(cfg, summarizePrompt(expanded))
    return HopResult(seed = seed, expandedEmail = expanded, summary = summary)
}

val hops = mutableListOf<HopResult>()

kotlinx.coroutines.runBlocking {
    repeat(30) { i ->
        val hop = runOneHop(seed, cfg)
        hops += hop
        println(hop)
        seed = hop.summary
        println("#${i + 1}: $seed")
    }
}

val last = hops.last()

println("\n=== FINAL SEED ===\n$seed\n")
println("=== LAST EXPANDED ===\n${last.expandedEmail}\n")
println("=== LAST SUMMARY ===\n${last.summary}\n")

HopResult(seed=It rained all day and we didn't get any work done., expandedEmail=Subject: When the Rain Hits, So Does the Pause Button ☔️

Hey Team,

Hope you’re all doing well! Just wanted to give you a quick update from our end. Today turned into one of those classic “rainy day” vibes — it poured nonstop from morning till evening, and honestly, it kind of put a damper on our productivity. With the weather being so relentless, we didn’t manage to get much work done outside as planned.

But hey, sometimes Mother Nature has other ideas, right? We took the opportunity to hit the pause button, recharge a bit, and brainstorm some fresh ideas indoors. Think of it as a creative reset — sometimes the best breakthroughs come when you step back and let the storm pass.

Looking ahead, we’re gearing up to dive back in full force tomorrow, rain or shine. Fingers crossed for clearer skies and a super productive day ahead!

Hope your day was a bit sunnier than ours. Catch you all soon!

Cheers,  
[Y

org.jetbrains.kotlinx.jupyter.exceptions.ReplInterruptedException: The execution was interrupted